In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/antycaptcha
!pip install tensorflow scikit-learn

/content/drive/MyDrive/antycaptcha


In [3]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split

In [4]:
from tensorflow.keras.utils import Sequence

class PairsTFRecordDataset(Sequence):
    def __init__(self, tfrecord_path, batch_size=8, img_size=(64, 40)):
        super().__init__()
        self.tfrecord_path = tfrecord_path
        self.batch_size = batch_size
        self.img_size = img_size

        feature_description = {
            'image1': tf.io.FixedLenFeature([], tf.string),
            'image2': tf.io.FixedLenFeature([], tf.string),
            'label': tf.io.FixedLenFeature([], tf.int64),
        }

        def _parse_example(example_proto):
            features = tf.io.parse_single_example(example_proto, feature_description)
            img1 = tf.io.decode_jpeg(features['image1'], channels=3)
            img2 = tf.io.decode_jpeg(features['image2'], channels=3)
            label = features['label']

            img1 = tf.image.resize(img1, self.img_size)
            img2 = tf.image.resize(img2, self.img_size)

            img1 = tf.cast(img1, tf.float32) / 255.0
            img2 = tf.cast(img2, tf.float32) / 255.0

            return img1, img2, label

        # Load all data into memory
        dataset = tf.data.TFRecordDataset(self.tfrecord_path)
        dataset = dataset.map(_parse_example, num_parallel_calls=tf.data.AUTOTUNE)
        images1, images2, labels = [], [], []
        for img1, img2, label in dataset:
            images1.append(img1.numpy())
            images2.append(img2.numpy())
            labels.append(label.numpy())

        self.images1 = np.stack(images1)
        self.images2 = np.stack(images2)
        self.labels = np.array(labels)
        self.num_examples = len(self.labels)
        self.permutation = np.arange(self.num_examples)

    def __len__(self):
        return int(np.ceil(self.num_examples / self.batch_size))

    def __getitem__(self, idx):
        # Shuffle is handled in on_epoch_end
        batch_indexes = self.permutation[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_images1 = self.images1[batch_indexes]
        batch_images2 = self.images2[batch_indexes]
        batch_labels = self.labels[batch_indexes]
        return {'input_a': batch_images1, 'input_b': batch_images2}, batch_labels

    def on_epoch_end(self):
        np.random.shuffle(self.permutation)

In [5]:
def build_encoder(input_shape=(64, 40, 3)):
    # Jeśli masz grayscale, zmień (40, 64, 1)
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(x)
    return Model(inputs, x, name="encoder")

In [6]:
def build_siamese_model(input_shape=(64, 40, 3)):
    encoder = build_encoder(input_shape)
    input_a = layers.Input(shape=input_shape, name='input_a')
    input_b = layers.Input(shape=input_shape, name='input_b')

    emb_a = encoder(input_a)
    emb_b = encoder(input_b)

    diff = layers.Lambda(
        lambda tensors: tf.abs(tensors[0] - tensors[1]),
        output_shape=(64,)
    )([emb_a, emb_b])
    x = layers.Dense(32, activation='relu')(diff)
    x = layers.Dense(1, activation='sigmoid')(x)  # Wyjście: prawdopodobieństwo tej samej postaci

    return Model([input_a, input_b], x)

splittowanie oryginalnego zbioru na treningowy i testowy

In [23]:
import tensorflow as tf
import numpy as np

tfrecord_path = "dataset/pairs.tfrecord"
n = 153000
indices = np.arange(n)
np.random.shuffle(indices)
split_point = int(0.85 * n)
train_indices = set(indices[:split_point])
test_indices = set(indices[split_point:])

with tf.io.TFRecordWriter("dataset/train_dataset.tfrecord") as train_writer, \
     tf.io.TFRecordWriter("dataset/test_dataset.tfrecord") as test_writer:

    for i, record in enumerate(tf.data.TFRecordDataset(tfrecord_path)):
        if i in train_indices:
            train_writer.write(record.numpy())
        else:
            test_writer.write(record.numpy())

ladowanie obu zbiorow

In [ ]:
##%%
train_path = os.path.join('dataset', 'train_dataset.tfrecord')
test_path = os.path.join('dataset', 'test_dataset.tfrecord')

train_dataset = PairsTFRecordDataset(train_path)
test_dataset = PairsTFRecordDataset(test_path)

dlugosc zbiorow

In [24]:
print("Original:", len(list(tf.data.TFRecordDataset("dataset/pairs.tfrecord"))))
print("Train:", len(list(tf.data.TFRecordDataset("dataset/train_dataset.tfrecord"))))
print("Test:", len(list(tf.data.TFRecordDataset("dataset/test_dataset.tfrecord"))))

Original: 153000
Train: 130050
Test: 22950


trening modelu

In [ ]:
# from tensorflow.keras.callbacks import EarlyStopping

# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=3,
#     restore_best_weights=True
# )

model = build_siamese_model(input_shape=(64, 40, 3))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=3,
    # callbacks=[early_stop]
)

model.save('compare_two_modelv2.1.keras')
print('Model saved to compare_two_modelv2.1.keras')



Epoch 1/3
16257/16257 ━━━━━━━━━━━━━━━━━━━━ 680s 42ms/step - accuracy: 0.9264 - loss: 0.1817 - val_accuracy: 0.9861 - val_loss: 0.0417
Epoch 2/3
16257/16257 ━━━━━━━━━━━━━━━━━━━━ 676s 42ms/step - accuracy: 0.9844 - loss: 0.0483 - val_accuracy: 0.9908 - val_loss: 0.0306
Epoch 3/3
 3682/16257 ━━━━━━━━━━━━━━━━━━━━ 8:09 39ms/step - accuracy: 0.9902 - loss: 0.0322